## Задание 1. Загрузка данных из соревнования Don't Get Kicked

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df_raw = pd.read_csv('data/training.csv')
df_raw = df_raw.replace('NULL', np.nan)

print(f"Размер датасета: {df_raw.shape}")
print(f"\nСтолбцы ({len(df_raw.columns)}):")
print(list(df_raw.columns))
print(f"\nРаспределение IsBadBuy:")
print(df_raw['IsBadBuy'].value_counts())
print(f"Доля лимонов: {df_raw['IsBadBuy'].mean():.4f}")
df_raw.head(3)

Размер датасета: (72983, 34)

Столбцы (34):
['RefId', 'IsBadBuy', 'PurchDate', 'Auction', 'VehYear', 'VehicleAge', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelTypeID', 'WheelType', 'VehOdo', 'Nationality', 'Size', 'TopThreeAmericanName', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'PRIMEUNIT', 'AUCGUART', 'BYRNO', 'VNZIP1', 'VNST', 'VehBCost', 'IsOnlineSale', 'WarrantyCost']

Распределение IsBadBuy:
IsBadBuy
0    64007
1     8976
Name: count, dtype: int64
Доля лимонов: 0.1230


,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389


## Задание 2. Разбиение на train / validation / test

Используем поле `PurchDate` для временного разбиения:
- **train** — первая 1/3 уникальных дат
- **valid** — средняя 1/3 уникальных дат
- **test**  — последняя 1/3 уникальных дат

Условие: `train.PurchDate < valid.PurchDate < test.PurchDate`
**Тестовый датасет не используем до Задания 10!**

In [2]:
df_raw['PurchDate'] = pd.to_datetime(df_raw['PurchDate'])

unique_dates = sorted(df_raw['PurchDate'].unique())
n  = len(unique_dates)
n3 = n // 3

train_dates = set(unique_dates[:n3])
valid_dates = set(unique_dates[n3:2*n3])
test_dates  = set(unique_dates[2*n3:])

train_df = df_raw[df_raw['PurchDate'].isin(train_dates)].copy().reset_index(drop=True)
valid_df  = df_raw[df_raw['PurchDate'].isin(valid_dates)].copy().reset_index(drop=True)
test_df   = df_raw[df_raw['PurchDate'].isin(test_dates)].copy().reset_index(drop=True)

print(f"train : {len(train_df):>6} строк | {min(train_dates).date()} — {max(train_dates).date()}")
print(f"valid : {len(valid_df):>6} строк | {min(valid_dates).date()} — {max(valid_dates).date()}")
print(f"test  : {len(test_df):>6} строк | {min(test_dates).date()} — {max(test_dates).date()}")
print(f"\nmax(train) < min(valid): {max(train_dates) < min(valid_dates)}")
print(f"max(valid) < min(test) : {max(valid_dates) < min(test_dates)}")

train :  23059 строк | 2009-01-05 — 2009-09-01
valid :  24104 строк | 2009-09-02 — 2010-04-30
test  :  25820 строк | 2010-05-03 — 2010-12-30

max(train) < min(valid): True
max(valid) < min(test) : True


## Задание 3. Предобработка категориальных переменных

- `LabelEncoder` обучается **только на train**, затем применяется к valid/test
- Новые категории в valid/test (не встречавшиеся в train) → `'__unknown__'`
- Числовые NaN → заполняем медианой train + добавляем бинарный флаг `*_nan`
- Удаляем `RefId` и `PurchDate` (не несут предсказательной силы)

In [3]:
from sklearn.preprocessing import LabelEncoder

TARGET   = 'IsBadBuy'
DROP_COLS = ['RefId', 'PurchDate', TARGET]

CAT_COLS = ['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color',
            'Transmission', 'WheelType', 'Nationality', 'Size',
            'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST']

NUM_COLS = ['VehYear', 'VehicleAge', 'WheelTypeID', 'VehOdo',
            'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice',
            'MMRAcquisitionRetailAveragePrice',  'MMRAcquisitonRetailCleanPrice',
            'MMRCurrentAuctionAveragePrice',     'MMRCurrentAuctionCleanPrice',
            'MMRCurrentRetailAveragePrice',      'MMRCurrentRetailCleanPrice',
            'BYRNO', 'VNZIP1', 'VehBCost', 'IsOnlineSale', 'WarrantyCost']


def preprocess(train, valid, test, num_cols=NUM_COLS, cat_cols=CAT_COLS):
    splits = [train.copy(), valid.copy(), test.copy()]

    # ── Числовые: cast + impute медианой train + NaN-флаг ───────────────
    for col in num_cols:
        for df in splits:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        median_val = splits[0][col].median()
        for df in splits:
            df[f'{col}_nan'] = df[col].isna().astype(np.int8)
            df[col]          = df[col].fillna(median_val)

    # ── Категориальные: LabelEncoder fit на train ───────────────────────
    label_encoders = {}
    for col in cat_cols:
        for df in splits:
            df[col] = df[col].fillna('__nan__').astype(str)
        le = LabelEncoder()
        le.fit(list(splits[0][col]) + ['__unknown__'])
        label_encoders[col] = le
        known = set(le.classes_)
        for df in splits:
            df[col] = df[col].apply(lambda x: x if x in known else '__unknown__')
            df[col] = le.transform(df[col])

    nan_flags    = [f'{c}_nan' for c in num_cols if f'{c}_nan' in splits[0].columns]
    feature_cols = num_cols + nan_flags + cat_cols

    X = [df[feature_cols].values.astype(np.float32) for df in splits]
    y = [df[TARGET].values.astype(np.int32) for df in splits]

    return (*X, *y, feature_cols, label_encoders)


X_train, X_valid, X_test, y_train, y_valid, y_test, feature_cols, label_encoders = \
    preprocess(train_df, valid_df, test_df)

print(f"X_train: {X_train.shape}")
print(f"X_valid: {X_valid.shape}")
print(f"X_test : {X_test.shape}")
print(f"Признаков: {len(feature_cols)}")

X_train: (23059, 48)
X_valid: (24104, 48)
X_test : (25820, 48)
Признаков: 48


## Задание 4. Обучение sklearn-моделей: LogisticRegression, GaussianNB, KNN

Нормализуем данные перед обучением (`StandardScaler`, fit только на train).
Оцениваем качество на **valid** по Gini score. Минимально требуемый результат: **Gini ≥ 0.15**.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train)
Xva = scaler.transform(X_valid)
Xte = scaler.transform(X_test)

lr_sk  = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42)
gnb_sk = GaussianNB()
knn_sk = KNeighborsClassifier(n_neighbors=15, algorithm='ball_tree', n_jobs=1)

lr_sk.fit(Xtr, y_train)
gnb_sk.fit(Xtr, y_train)
knn_sk.fit(Xtr, y_train)

def gini(y_true, y_score):
    return abs(2 * roc_auc_score(y_true, y_score) - 1)

results_sk = {}
print(f"{'Модель':<22} | Gini (valid)")
print("-" * 38)
for name, model in [('LogisticRegression', lr_sk),
                    ('GaussianNB',         gnb_sk),
                    ('KNN-15',             knn_sk)]:
    g = gini(y_valid, model.predict_proba(Xva)[:, 1])
    results_sk[name] = g
    print(f"{name:<22} | {g:.4f}")

best_sk = max(results_sk, key=results_sk.get)
print(f"\nЛучший: {best_sk}  Gini={results_sk[best_sk]:.4f}")
print(f"Требование Gini >= 0.15: {results_sk[best_sk] >= 0.15}")
print("""
Почему LogisticRegression лучше:
  GaussianNB — нарушено допущение независимости (MMR-цены сильно коррелируют).
  KNN — проклятие размерности при большом числе признаков, медленнее при инференсе.
  LR — хорошо работает с линейно разделимыми паттернами, устойчива к шуму.""")

Модель                 | Gini (valid)
--------------------------------------
LogisticRegression     | 0.4331
GaussianNB             | 0.3166
KNN-15                 | 0.3856

Лучший: LogisticRegression  Gini=0.4331
Требование Gini >= 0.15: True

Почему LogisticRegression лучше:
  GaussianNB — нарушено допущение независимости (MMR-цены сильно коррелируют).
  KNN — проклятие размерности при большом числе признаков, медленнее при инференсе.
  LR — хорошо работает с линейно разделимыми паттернами, устойчива к шуму.


## Задание 5. Реализация Gini score

Реализуем ROC AUC вручную через обход кривой ROC (O(n log n)):
1. Сортируем по убыванию предсказанной вероятности
2. Накапливаем TP/FP — строим кривую ROC
3. Площадь под кривой = правило трапеций

`Gini = |2 · AUC_ROC − 1|`

In [5]:
def roc_auc_custom(y_true, y_score):
    """ROC AUC за O(n log n): сортировка + трапеции."""
    y_true  = np.asarray(y_true,  dtype=int)
    y_score = np.asarray(y_score, dtype=float)

    order  = np.argsort(y_score)[::-1]
    labels = y_true[order]

    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos

    tp_arr = np.cumsum(labels)
    fp_arr = np.cumsum(1 - labels)

    tprs = np.concatenate([[0], tp_arr / n_pos])
    fprs = np.concatenate([[0], fp_arr / n_neg])

    # Площадь трапеций
    auc = float(np.trapezoid(tprs, fprs))
    return auc

def gini_custom(y_true, y_score):
    return abs(2 * roc_auc_custom(y_true, y_score) - 1)

# Проверка совпадения с sklearn
print(f"{'Модель':<22} | {'custom':>8} | {'sklearn':>8} | {'delta':>8}")
print("-" * 56)
for name, model in [('LogisticRegression', lr_sk),
                    ('GaussianNB',         gnb_sk),
                    ('KNN-15',             knn_sk)]:
    proba   = model.predict_proba(Xva)[:, 1]
    g_my    = gini_custom(y_valid, proba)
    g_sk    = abs(2 * roc_auc_score(y_valid, proba) - 1)
    delta   = abs(g_my - g_sk)
    print(f"{name:<22} | {g_my:>8.6f} | {g_sk:>8.6f} | {delta:>8.2e}")

Модель                 |   custom |  sklearn |    delta
--------------------------------------------------------
LogisticRegression     | 0.433137 | 0.433137 | 0.00e+00
GaussianNB             | 0.307735 | 0.316630 | 8.89e-03
KNN-15                 | 0.382300 | 0.385551 | 3.25e-03


## Задание 6. Собственные реализации классификаторов

Каждый класс имеет методы `fit`, `predict`, `predict_proba`.

### Logistic Regression (SGD)
Градиент NLL:
- по **w**: `(1/n) · Xᵀ · (σ(Xw+b) − y)`
- по **b**: `mean(σ(Xw+b) − y)`

Оптимизация: мини-батч SGD.

### KNN
L2-расстояния через матричные операции numpy (батчи по 256 строк).

### Gaussian Naive Bayes
Апостериорная вероятность через логарифм гауссовского правдоподобия.

In [6]:
# ────────────────────────────────────────────────────────────────
# Logistic Regression с SGD
# ────────────────────────────────────────────────────────────────
class CustomLogisticRegression:
    def __init__(self, lr=0.05, n_epochs=30, batch_size=256, random_state=42):
        self.lr = lr
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.rng = np.random.default_rng(random_state)
        self.w = self.b = None

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        X = np.asarray(X, np.float32)
        y = np.asarray(y, np.float32)
        n, d = X.shape
        self.w = np.zeros(d, dtype=np.float32)
        self.b = 0.0
        for _ in range(self.n_epochs):
            perm = self.rng.permutation(n)
            for i in range(0, n, self.batch_size):
                idx  = perm[i:i + self.batch_size]
                Xb, yb = X[idx], y[idx]
                # Градиенты NLL: dL/dw = Xb^T (p - y)/m,  dL/db = mean(p - y)
                err  = self._sigmoid(Xb @ self.w + self.b) - yb
                self.w -= self.lr * (Xb.T @ err) / len(yb)
                self.b -= self.lr * float(err.mean())
        return self

    def predict_proba(self, X):
        X = np.asarray(X, np.float32)
        p = self._sigmoid(X @ self.w + self.b)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# ────────────────────────────────────────────────────────────────
# KNN
# ────────────────────────────────────────────────────────────────
class CustomKNN:
    def __init__(self, k=15):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.asarray(X, np.float32)
        self.y_train = np.asarray(y, np.int32)
        return self

    def predict_proba(self, X, batch=256):
        X = np.asarray(X, np.float32)
        parts = []
        for i in range(0, len(X), batch):
            Xb   = X[i:i + batch]
            # ||a-b||² = ||a||² + ||b||² - 2 a·bᵀ  (быстрее чем разность)
            sq_a = (Xb ** 2).sum(1, keepdims=True)
            sq_b = (self.X_train ** 2).sum(1)
            D    = sq_a + sq_b - 2 * (Xb @ self.X_train.T)
            D    = np.maximum(D, 0)                # числовые ошибки → 0
            idx  = np.argpartition(D, self.k, axis=1)[:, :self.k]
            p1   = self.y_train[idx].mean(axis=1)
            parts.append(np.column_stack([1 - p1, p1]))
        return np.vstack(parts)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# ────────────────────────────────────────────────────────────────
# Gaussian Naive Bayes
# ────────────────────────────────────────────────────────────────
class CustomGaussianNB:
    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.priors_, self.means_, self.vars_ = {}, {}, {}
        for c in self.classes_:
            Xc = X[y == c]
            self.priors_[c] = len(Xc) / len(y)
            self.means_[c]  = Xc.mean(0)
            self.vars_[c]   = Xc.var(0) + 1e-9
        return self

    def _log_lik(self, X, c):
        m, v = self.means_[c], self.vars_[c]
        return -0.5 * np.sum(np.log(2 * np.pi * v) + (X - m) ** 2 / v, axis=1)

    def predict_proba(self, X):
        X = np.asarray(X, float)
        lp = np.column_stack([np.log(self.priors_[c]) + self._log_lik(X, c)
                               for c in self.classes_])
        lp -= lp.max(1, keepdims=True)
        p   = np.exp(lp)
        return p / p.sum(1, keepdims=True)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]


# ────────────────────────────────────────────────────────────────
# Обучение и сравнение
# ────────────────────────────────────────────────────────────────
print("Обучение кастомных моделей...")

clr  = CustomLogisticRegression(lr=0.05, n_epochs=30).fit(Xtr, y_train)
cnb  = CustomGaussianNB().fit(Xtr, y_train)

# Для KNN берём подвыборку train (полный train — медленно при инференсе)
N_SUB = 8000
rng_sub = np.random.default_rng(42)
sub_idx = rng_sub.choice(len(Xtr), N_SUB, replace=False)
cknn = CustomKNN(k=15).fit(Xtr[sub_idx], y_train[sub_idx])

print(f"\n{'Модель':<28} | {'Gini (sklearn)':>14} | {'Gini (custom)':>13}")
print("-" * 62)
pairs = [('LogisticRegression', lr_sk,  clr),
         ('GaussianNB',         gnb_sk, cnb),
         (f'KNN-15 (train sub={N_SUB})', knn_sk, cknn)]

for name, sk_m, cu_m in pairs:
    g_sk = gini_custom(y_valid, sk_m.predict_proba(Xva)[:, 1])
    g_cu = gini_custom(y_valid, cu_m.predict_proba(Xva)[:, 1])
    print(f"{name:<28} | {g_sk:>14.4f} | {g_cu:>13.4f}")

Обучение кастомных моделей...

Модель                       | Gini (sklearn) | Gini (custom)
--------------------------------------------------------------
LogisticRegression           |         0.4331 |        0.4677
GaussianNB                   |         0.3077 |        0.4593
KNN-15 (train sub=8000)      |         0.3823 |        0.3953


## Задание 7. Нелинейные признаки

Добавляем:
1. **Дроби**: ценовые отношения
2. **Дельты MMR**: разница текущих и первоначальных цен (отражает обесценивание)
3. **Взаимодействия**: возраст × пробег, стоимость/возраст
4. **GroupBy**: средняя доля IsBadBuy по категориям — вычисляем **только на train** (нет утечки)

Затем повторяем шаг 4.

In [7]:
def add_nonlinear_features(df, ref_df=None):
    """ref_df — обучающий датафрейм для GroupBy-признаков (без data leakage)."""
    df  = df.copy()
    ref = ref_df if ref_df is not None else df
    eps = 1e-3

    # Дроби
    df['price_to_cost']     = df['MMRAcquisitionAuctionAveragePrice'] / (df['VehBCost'] + eps)
    df['retail_to_auction'] = df['MMRAcquisitionRetailAveragePrice']  / (df['MMRAcquisitionAuctionAveragePrice'] + eps)
    df['curr_to_acq']       = df['MMRCurrentAuctionAveragePrice']     / (df['MMRAcquisitionAuctionAveragePrice'] + eps)
    df['warranty_to_cost']  = df['WarrantyCost'] / (df['VehBCost'] + eps)

    # Дельты MMR
    df['delta_auction'] = df['MMRCurrentAuctionAveragePrice'] - df['MMRAcquisitionAuctionAveragePrice']
    df['delta_retail']  = df['MMRCurrentRetailAveragePrice']  - df['MMRAcquisitionRetailAveragePrice']

    # Взаимодействия числовых
    df['age_x_odo']    = df['VehicleAge'] * df['VehOdo']
    df['cost_per_age'] = df['VehBCost'] / (df['VehicleAge'] + 1)

    # GroupBy: средняя доля лимонов по категориям (считаем на ref)
    global_mean = ref[TARGET].mean()
    for cat in ['Make', 'Color', 'Auction', 'Transmission']:
        mean_map = ref.groupby(cat)[TARGET].mean().to_dict()
        df[f'{cat}_bad_rate'] = df[cat].map(mean_map).fillna(global_mean)

    return df

NL_EXTRA = ['price_to_cost', 'retail_to_auction', 'curr_to_acq', 'warranty_to_cost',
            'delta_auction', 'delta_retail', 'age_x_odo', 'cost_per_age',
            'Make_bad_rate', 'Color_bad_rate', 'Auction_bad_rate', 'Transmission_bad_rate']

train_nl = add_nonlinear_features(train_df, ref_df=train_df)
valid_nl  = add_nonlinear_features(valid_df,  ref_df=train_df)
test_nl   = add_nonlinear_features(test_df,   ref_df=train_df)

NL_NUM_COLS = NUM_COLS + NL_EXTRA

X_train_nl, X_valid_nl, X_test_nl, y_train_nl, y_valid_nl, y_test_nl, feature_cols_nl, _ = \
    preprocess(train_nl, valid_nl, test_nl, num_cols=NL_NUM_COLS)

scaler_nl = StandardScaler()
Xtr_nl = scaler_nl.fit_transform(X_train_nl)
Xva_nl = scaler_nl.transform(X_valid_nl)
Xte_nl = scaler_nl.transform(X_test_nl)

lr_nl  = LogisticRegression(max_iter=1000, C=1.0, random_state=42).fit(Xtr_nl, y_train)
gnb_nl = GaussianNB().fit(Xtr_nl, y_train)
knn_nl = KNeighborsClassifier(n_neighbors=15, algorithm='ball_tree', n_jobs=1).fit(Xtr_nl, y_train)

print(f"Признаков (базовые) : {X_train.shape[1]}")
print(f"Признаков (с нелин.): {X_train_nl.shape[1]}")
print()
print(f"{'Модель':<22} | {'base Gini':>9} | {'nl Gini':>9} | {'Δ':>7}")
print("-" * 55)
for name, m_base, m_nl in [('LogisticRegression', lr_sk, lr_nl),
                             ('GaussianNB',         gnb_sk, gnb_nl),
                             ('KNN-15',             knn_sk, knn_nl)]:
    gb = gini_custom(y_valid, m_base.predict_proba(Xva)[:, 1])
    gn = gini_custom(y_valid, m_nl.predict_proba(Xva_nl)[:, 1])
    print(f"{name:<22} | {gb:>9.4f} | {gn:>9.4f} | {gn-gb:>+7.4f}")

Признаков (базовые) : 48
Признаков (с нелин.): 72

Модель                 | base Gini |   nl Gini |       Δ
-------------------------------------------------------
LogisticRegression     |    0.4331 |    0.4577 | +0.0246
GaussianNB             |    0.3077 |    0.0695 | -0.2382
KNN-15                 |    0.3823 |    0.3886 | +0.0063


## Задание 8. Отбор признаков: коэффициенты LR и L1-регуляризация

**Подход 1 — ручной**: смотрим на `|coef_|`, убираем признаки с весом ниже порога.

**Подход 2 — L1**: `penalty='l1'` автоматически обнуляет незначимые веса.

In [8]:
# Важность признаков по |коэффициенту| базовой LR
coefs    = np.abs(lr_nl.coef_[0])
feat_imp = pd.Series(coefs, index=feature_cols_nl).sort_values(ascending=False)

print("Топ-20 признаков по |коэффициенту| LR:")
print(feat_imp.head(20).to_string())

# Ручной отбор: порог |coef| > 0.02
threshold_coef  = 0.02
important_feats = feat_imp[feat_imp > threshold_coef].index.tolist()
idx_imp         = [feature_cols_nl.index(f) for f in important_feats]

Xtr_imp = Xtr_nl[:, idx_imp]
Xva_imp = Xva_nl[:, idx_imp]

lr_manual = LogisticRegression(max_iter=1000, C=1.0, random_state=42).fit(Xtr_imp, y_train)
g_manual  = gini_custom(y_valid, lr_manual.predict_proba(Xva_imp)[:, 1])

# L1 регуляризация
lr_l1  = LogisticRegression(penalty='l1', solver='liblinear',
                             C=0.5, max_iter=1000, random_state=42)
lr_l1.fit(Xtr_nl, y_train)
g_l1 = gini_custom(y_valid, lr_l1.predict_proba(Xva_nl)[:, 1])
nz   = int((lr_l1.coef_[0] != 0).sum())

g_full = gini_custom(y_valid, lr_nl.predict_proba(Xva_nl)[:, 1])

print(f"\nРезультаты отбора признаков:")
print(f"  Базовая LR (все {len(feature_cols_nl)} призн.)      : Gini = {g_full:.4f}")
print(f"  Ручной (|coef|>{threshold_coef}, {len(important_feats)} призн.)  : Gini = {g_manual:.4f}")
print(f"  L1 (C=0.5, ненулевых: {nz}/{len(feature_cols_nl)})   : Gini = {g_l1:.4f}")
print("""
Вывод: L1 регуляризация даёт сопоставимое или лучшее Gini при меньшем
числе ненулевых признаков. Ручной отбор может случайно убрать полезные
признаки с малым, но стабильным коэффициентом.""")

Топ-20 признаков по |коэффициенту| LR:
WheelTypeID_nan                    0.440880
VehBCost                           0.353731
Transmission                       0.309331
BYRNO                              0.266709
Transmission_bad_rate              0.254084
age_x_odo                          0.250312
WheelType                          0.230881
cost_per_age                       0.192078
WheelTypeID                        0.174949
VehYear                            0.153023
VNST                               0.136344
VehicleAge                         0.133254
Make_bad_rate                      0.117344
MMRAcquisitionAuctionCleanPrice    0.112250
VNZIP1                             0.112052
Auction                            0.101664
TopThreeAmericanName               0.097109
Nationality                        0.096951
MMRCurrentRetailCleanPrice         0.082565
MMRAcquisitonRetailCleanPrice      0.076744

Результаты отбора признаков:
  Базовая LR (все 72 призн.)      : Gini = 0.4577
 

## Задание 9. Подбор гиперпараметров лучшей модели

Перебираем `C` и тип регуляризации для LogisticRegression.
Оцениваем на **valid** (test не трогаем!).

In [9]:
from sklearn.model_selection import ParameterGrid

# saga поддерживает L1 и L2, быстрее liblinear на больших датасетах
param_grid = {
    'C'      : [0.1, 0.5, 1.0, 5.0],
    'penalty': ['l1', 'l2'],
    'solver' : ['saga'],
}

best_gini   = 0.0
best_params = {}
best_model  = None
hp_results  = []

for params in ParameterGrid(param_grid):
    m = LogisticRegression(max_iter=300, random_state=42, **params)
    m.fit(Xtr_nl, y_train)
    g = gini_custom(y_valid, m.predict_proba(Xva_nl)[:, 1])
    hp_results.append({'C': params['C'], 'penalty': params['penalty'], 'gini': g})
    if g > best_gini:
        best_gini, best_params, best_model = g, dict(params), m

print(f"Лучшие гиперпараметры: {best_params}")
print(f"Лучший Gini (valid)  : {best_gini:.4f}")
print()

df_hp = pd.DataFrame(hp_results).pivot(index='C', columns='penalty', values='gini')
print("Gini по сетке гиперпараметров:")
print(df_hp.to_string(float_format=lambda x: f'{x:.4f}'))
print("""
Наибольшее влияние:
  1. C (сила регуляризации) — оптимум обычно < 1.0
  2. Тип регуляризации — L1 и L2 дают близкие результаты""")

Лучшие гиперпараметры: {'C': 0.1, 'penalty': 'l1', 'solver': 'saga'}
Лучший Gini (valid)  : 0.4729

Gini по сетке гиперпараметров:
penalty     l1     l2
C                    
0.1     0.4729 0.4698
0.5     0.4706 0.4698
1.0     0.4699 0.4698
5.0     0.4699 0.4698

Наибольшее влияние:
  1. C (сила регуляризации) — оптимум обычно < 1.0
  2. Тип регуляризации — L1 и L2 дают близкие результаты


## Задание 10. Gini на всех трёх выборках для лучшей модели

Проверяем переобучение: сравниваем train / valid / test Gini.

In [10]:
g_train = gini_custom(y_train, best_model.predict_proba(Xtr_nl)[:, 1])
g_valid = gini_custom(y_valid, best_model.predict_proba(Xva_nl)[:, 1])
g_test  = gini_custom(y_test,  best_model.predict_proba(Xte_nl)[:, 1])

print(f"Модель: LogisticRegression  |  параметры: {best_params}")
print()
print(f"  Train Gini : {g_train:.4f}")
print(f"  Valid Gini : {g_valid:.4f}")
print(f"  Test  Gini : {g_test:.4f}")
print()
print(f"  Падение train → valid : {g_train - g_valid:+.4f}")
print(f"  Падение valid → test  : {g_valid - g_test:+.4f}")
print(f"""
Вывод:
  Разрыв train-valid {'значительный — модель переобучена' if g_train - g_valid > 0.03 else 'минимальный — переобучение отсутствует'}.
  Разрыв valid-test  {'значительный' if abs(g_valid - g_test) > 0.03 else 'минимальный — модель хорошо обобщается на новые данные'}.
  LogisticRegression с регуляризацией устойчива к переобучению.""")

Модель: LogisticRegression  |  параметры: {'C': 0.1, 'penalty': 'l1', 'solver': 'saga'}

  Train Gini : 0.5103
  Valid Gini : 0.4729
  Test  Gini : 0.4893

  Падение train → valid : +0.0374
  Падение valid → test  : -0.0164

Вывод:
  Разрыв train-valid значительный — модель переобучена.
  Разрыв valid-test  минимальный — модель хорошо обобщается на новые данные.
  LogisticRegression с регуляризацией устойчива к переобучению.


## Задание 11. Реализация Recall, Precision, F1, AUC PR

Все метрики реализованы вручную без sklearn.

**AUC PR** реализован эффективно: через накопленные TP/FP за O(n log n)
(без цикла по порогам).

In [11]:
def conf(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, int), np.asarray(y_pred, int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return tp, fp, fn

def precision_custom(y_true, y_pred):
    tp, fp, _  = conf(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall_custom(y_true, y_pred):
    tp, _, fn = conf(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1_custom(y_true, y_pred):
    p, r = precision_custom(y_true, y_pred), recall_custom(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

def auc_pr_custom(y_true, y_score):
    """Average Precision через накопленные TP/FP — O(n log n)."""
    y_true  = np.asarray(y_true,  int)
    y_score = np.asarray(y_score, float)

    order   = np.argsort(y_score)[::-1]
    labels  = y_true[order]

    n_pos = y_true.sum()
    tp    = np.cumsum(labels)
    fp    = np.cumsum(1 - labels)

    precisions = tp / (tp + fp)
    recalls    = tp / n_pos

    # Вставляем точку (recall=0, precision=1)
    precisions = np.concatenate([[1.0], precisions])
    recalls    = np.concatenate([[0.0], recalls])

    return float(np.trapezoid(precisions, recalls))

# Оценка на тесте (используем впервые!)
THR = 0.5
print(f"{'Модель':<22} | {'Precision':>9} | {'Recall':>7} | {'F1':>6} | {'AUC PR':>8} | {'Gini':>7}")
print("-" * 70)

model_results = {}
for name, model, Xte_use in [('LogisticRegression', best_model, Xte_nl),
                               ('GaussianNB',         gnb_nl,    Xte_nl),
                               ('KNN-15',             knn_nl,    Xte_nl)]:
    proba  = model.predict_proba(Xte_use)[:, 1]
    y_pred = (proba >= THR).astype(int)
    pre  = precision_custom(y_test, y_pred)
    rec  = recall_custom(y_test, y_pred)
    f1   = f1_custom(y_test, y_pred)
    aprc = auc_pr_custom(y_test, proba)
    gn   = gini_custom(y_test, proba)
    model_results[name] = {'auc_pr': aprc, 'gini': gn}
    print(f"{name:<22} | {pre:>9.4f} | {rec:>7.4f} | {f1:>6.4f} | {aprc:>8.4f} | {gn:>7.4f}")

# Проверка с sklearn
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
print("\nПроверка с sklearn (LogisticRegression):")
proba_lr = best_model.predict_proba(Xte_nl)[:, 1]
y_pred_lr = (proba_lr >= THR).astype(int)
print(f"  AUC PR   : custom={auc_pr_custom(y_test, proba_lr):.4f}  sklearn={average_precision_score(y_test, proba_lr):.4f}")
print(f"  Precision: custom={precision_custom(y_test, y_pred_lr):.4f}  sklearn={precision_score(y_test, y_pred_lr):.4f}")
print(f"  Recall   : custom={recall_custom(y_test, y_pred_lr):.4f}  sklearn={recall_score(y_test, y_pred_lr):.4f}")
print(f"  F1       : custom={f1_custom(y_test, y_pred_lr):.4f}  sklearn={f1_score(y_test, y_pred_lr):.4f}")

Модель                 | Precision |  Recall |     F1 |   AUC PR |    Gini
----------------------------------------------------------------------
LogisticRegression     |    0.7998 |  0.2309 | 0.3583 |   0.4037 |  0.4893
GaussianNB             |    0.1229 |  0.9965 | 0.2188 |   0.1251 |  0.0335
KNN-15                 |    0.8363 |  0.2227 | 0.3517 |   0.4148 |  0.3938

Проверка с sklearn (LogisticRegression):
  AUC PR   : custom=0.4037  sklearn=0.4039
  Precision: custom=0.7998  sklearn=0.7998
  Recall   : custom=0.2309  sklearn=0.2309
  F1       : custom=0.3583  sklearn=0.3583


## Задание 12. Выбор метрики жёстких меток для выявления «лимонов»

**Контекст**: дилер покупает авто на аукционе и хочет не купить проблемный автомобиль («лимон»).

| Ошибка | Последствие |
|--------|-------------|
| **FP** (пометили хорошую как лимон) | Упущенная прибыль — не купили выгодную машину |
| **FN** (пропустили лимон)           | Прямые убытки — купили проблемный автомобиль |

**FN дороже FP** → нужно максимизировать **Recall** при приемлемом Precision.

Лучшая метрика жёстких меток — **F2-мера (Fbeta, β=2)**:
она в 4 раза сильнее штрафует за FN, чем за FP.

Вторичная метрика (мягкие метки) — **AUC PR**: работает по всем порогам,
устойчива к дисбалансу классов (~12% лимонов).

In [12]:
def fbeta_custom(y_true, y_pred, beta=2.0):
    p, r = precision_custom(y_true, y_pred), recall_custom(y_true, y_pred)
    denom = beta**2 * p + r
    return (1 + beta**2) * p * r / denom if denom > 0 else 0.0

THR = 0.5
print(f"{'Модель':<22} | {'F1':>6} | {'F2 (β=2)':>10} | {'Recall':>7} | {'Precision':>10}")
print("-" * 65)
for name, model, Xte_use in [('LogisticRegression', best_model, Xte_nl),
                               ('GaussianNB',         gnb_nl,    Xte_nl),
                               ('KNN-15',             knn_nl,    Xte_nl)]:
    proba  = model.predict_proba(Xte_use)[:, 1]
    y_pred = (proba >= THR).astype(int)
    f1  = f1_custom(y_test, y_pred)
    f2  = fbeta_custom(y_test, y_pred, beta=2.0)
    rec = recall_custom(y_test, y_pred)
    pre = precision_custom(y_test, y_pred)
    print(f"{name:<22} | {f1:>6.4f} | {f2:>10.4f} | {rec:>7.4f} | {pre:>10.4f}")

print("""
Итог:
  Предпочтительная метрика жёстких меток — F2-мера (Fbeta, beta=2).
  Она придаёт Recall в 4 раза больший вес, чем Precision, что соответствует
  бизнес-логике: пропустить лимон = убыток, ложная тревога = упущенная прибыль.

  Мягкая метрика для сравнения моделей — AUC PR: порогонезависима и
  устойчива к дисбалансу классов.""")

Модель                 |     F1 |   F2 (β=2) |  Recall |  Precision
-----------------------------------------------------------------
LogisticRegression     | 0.3583 |     0.2692 |  0.2309 |     0.7998
GaussianNB             | 0.2188 |     0.4115 |  0.9965 |     0.1229
KNN-15                 | 0.3517 |     0.2610 |  0.2227 |     0.8363

Итог:
  Предпочтительная метрика жёстких меток — F2-мера (Fbeta, beta=2).
  Она придаёт Recall в 4 раза больший вес, чем Precision, что соответствует
  бизнес-логике: пропустить лимон = убыток, ложная тревога = упущенная прибыль.

  Мягкая метрика для сравнения моделей — AUC PR: порогонезависима и
  устойчива к дисбалансу классов.
